In [ ]:
!pip install sentence-transformers gensim pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 31.9 MB/s eta 0:00:00


First file to get the ratings

In [ ]:



import pandas as pd


df = pd.read_csv("absa_output_electric_tb_combined.csv")

print(df.columns)
df.head()

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def build_text(row):
    aspect = str(row.get('aspect', ''))
    opinion = str(row.get('opinion_term', ''))
    return (aspect + " " + opinion).strip()

df['text_for_analysis'] = df.apply(build_text, axis=1)

In [ ]:
scale = ["worst", "bad", "okay", "good", "excellent"]

scale_map = {
    "positive": ["good", "excellent"],
    "negative": ["bad", "worst"],
    "neutral": ["okay"]
}

SBERT

In [ ]:
from sentence_transformers import SentenceTransformer
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def map_sbert(text, sentiment):
    words = scale_map.get(str(sentiment).lower(), scale)

    scale_emb = sbert_model.encode(words)
    text_emb = sbert_model.encode([text])

    sims = cosine_similarity(text_emb, scale_emb)
    return words[np.argmax(sims)]

Word2Vec

In [ ]:
import gensim.downloader as api
w2v_model = api.load("word2vec-google-news-300")

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
def get_w2v_vector(text):
    words = str(text).lower().split()
    vecs = [w2v_model[w] for w in words if w in w2v_model]

    if len(vecs) == 0:
        return np.zeros(300)

    return np.mean(vecs, axis=0)

In [ ]:
def map_w2v(text, sentiment):
    words = scale_map.get(str(sentiment).lower(), scale)

    scale_vecs = np.array([get_w2v_vector(w) for w in words])
    text_vec = get_w2v_vector(text).reshape(1, -1)

    sims = cosine_similarity(text_vec, scale_vecs)
    return words[np.argmax(sims)]

Glove

In [ ]:
glove_model = api.load("glove-wiki-gigaword-300")

[==================================================] 100.0% 376.1/376.1MB downloaded


In [ ]:
def get_glove_vector(text):
    words = str(text).lower().split()
    vecs = [glove_model[w] for w in words if w in glove_model]

    if len(vecs) == 0:
        return np.zeros(300)

    return np.mean(vecs, axis=0)

In [ ]:
def map_glove(text, sentiment):
    words = scale_map.get(str(sentiment).lower(), scale)

    scale_vecs = np.array([get_glove_vector(w) for w in words])
    text_vec = get_glove_vector(text).reshape(1, -1)

    sims = cosine_similarity(text_vec, scale_vecs)
    return words[np.argmax(sims)]

First file output ratings

In [ ]:
df['sbert_bucket'] = df.apply(
    lambda x: map_sbert(x['text_for_analysis'], x['sentiment']), axis=1
)

df['w2v_bucket'] = df.apply(
    lambda x: map_w2v(x['text_for_analysis'], x['sentiment']), axis=1
)

df['glove_bucket'] = df.apply(
    lambda x: map_glove(x['text_for_analysis'], x['sentiment']), axis=1
)
# Save AFTER mapping
output_file = "final_bucket_output.csv"
df.to_csv(output_file, index=False)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)          # Increase width
pd.set_option('display.max_colwidth', None)   # Show full text
df['word_count'] = df['sentence'].astype(str).apply(lambda x: len(x.split()))
filtered_df = df[(df['word_count'] >= 5) & (df['word_count'] <= 15)]
filtered_df = filtered_df.reset_index(drop=True)
print(filtered_df.head(20))


      asin  review_id  star_rating                                                                                    sentence              aspect opinion_term sentiment  confidence       text_for_analysis sbert_bucket w2v_bucket glove_bucket  word_count
0   Neotia          1            5                                  Amazing coordination among all and outstanding management.        coordination      Amazing  Positive      0.9985    coordination Amazing    excellent  excellent    excellent           7
1   Neotia          1            5                They both have literally taken ownership and done beyond the level expected.           ownership        taken  Positive      0.9979         ownership taken    excellent       good         good          12
2   Neotia          1            5                    And Rahul, my goodness he treat the clients like his own family members.               Rahul        treat  Negative      0.5011             Rahul treat        worst        bad      

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)          # Increase width
pd.set_option('display.max_colwidth', None)   # Show full text

print(df.head())

     asin  review_id  star_rating                                                                                           sentence        aspect opinion_term sentiment  confidence     text_for_analysis sbert_bucket w2v_bucket glove_bucket
0  Neotia          1            5                                         Amazing coordination among all and outstanding management.  coordination      Amazing  Positive      0.9985  coordination Amazing    excellent  excellent    excellent
1  Neotia          1            5                       They both have literally taken ownership and done beyond the level expected.     ownership        taken  Positive      0.9979       ownership taken    excellent       good         good
2  Neotia          1            5  And the way Rajoshree managed a guest who was a tough one, absolutely stunning and de-escalating.     Rajoshree      managed  Negative      0.5217     Rajoshree managed        worst      worst          bad
3  Neotia          1            5  A

In [ ]:
#from google.colab import files
#files.download(output_file)
# ✅ ADD THIS RIGHT HERE (same cell)
label_to_score = {
    "worst": 1,
    "bad": 2,
    "okay": 3,
    "good": 4,
    "excellent": 5
}

df['sbert_bucket'] = df['sbert_bucket'].map(label_to_score)
df['w2v_bucket'] = df['w2v_bucket'].map(label_to_score)
df['glove_bucket'] = df['glove_bucket'].map(label_to_score)
df['avg_score'] = df[['sbert_bucket', 'w2v_bucket', 'glove_bucket']].mean(axis=1)
# Save AFTER mapping
output_file = "final_bucket_output.csv"
df.to_csv(output_file, index=False)

from google.colab import files
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Second file import for one hot encoding

Import file with clusters and add the flag values

In [ ]:
# from google.colab import files
# uploaded = files.upload()

# import pandas as pd

# file_name = list(uploaded.keys())[0]
# df = pd.read_csv(file_name)
# df['Flag'] = 1
# df['Brand'] = 'Brand1'
# df['Brand_Having'] = df['cluster_level_bottom'].astype(str) + '_Flag'


# print(df.columns)
# df.head()


Saving final_bucket_output_with_clusters.csv to final_bucket_output_with_clusters.csv
Index(['review_id', 'Brand ', 'star_rating', 'sentence', 'aspect',
       'opinion_term', 'sentiment', 'confidence', 'text_for_analysis',
       'sbert_bucket', 'w2v_bucket', 'glove_bucket', 'llm_cluster',
       'llm_label', 'hac_cluster', 'cluster_level_bottom', 'Flag', 'Brand',
       'Brand_Having'],
      dtype='object')


,review_id,Brand,star_rating,sentence,aspect,opinion_term,sentiment,confidence,text_for_analysis,sbert_bucket,w2v_bucket,glove_bucket,llm_cluster,llm_label,hac_cluster,cluster_level_bottom,Flag,Brand,Brand_Having
0,A2ROBO96G69ABZ,A,5,be careful around the gum lines.,gum lines,NaN,Negative,0.9620,gum lines nan,bad,bad,bad,C09,Cleaning Performance,11,Vague,1,Brand1,Vague_Flag
1,A2ROBO96G69ABZ,A,5,It also auto shuts off in 2 mins.,auto shuts,NaN,Neutral,0.5322,auto shuts nan,okay,okay,okay,C07,Timer & Auto Features,4,Auto,1,Brand1,Auto_Flag
2,A2ROBO96G69ABZ,A,5,Would recommend to others and beats any deals ...,deals,beats,Negative,0.8075,deals beats,bad,bad,bad,C11,Value & Warranty,8,Design & Deals,1,Brand1,Design & Deals_Flag
3,A3AQP2VX0TJQ73,B,4,The head are a bit smaller than they used to be.,head,smaller,Negative,0.9963,head smaller,worst,bad,bad,C01,Brush Head,2,Brush & Oral Contact,1,Brand1,Brush & Oral Contact_Flag
4,A3AQP2VX0TJQ73,B,4,The power is about the same.,power,same,Neutral,0.7562,power same,okay,okay,okay,C03,Power & Motor,1,Power,1,Brand1,Power_Flag


In [ ]:

label_to_score = {
    "worst": 1,
    "bad": 2,
    "okay": 3,
    "good": 4,
    "excellent": 5
}

df['sbert_bucket'] = df['sbert_bucket'].map(label_to_score)
df['w2v_bucket'] = df['w2v_bucket'].map(label_to_score)
df['glove_bucket'] = df['glove_bucket'].map(label_to_score)
df['avg_score'] = df[['sbert_bucket', 'w2v_bucket', 'glove_bucket']].mean(axis=1)


In [ ]:
# # Step 1: Rename columns (if needed for consistency)
# df = df.rename(columns={
#     'cluster_level_bottom': 'Aspect',
#     'avg_score': 'Avg rating'
# })

# # Step 2: One-hot encode Brand
# brand_ohe = pd.get_dummies(df['Brand'])

# # ✅ FIXED: Brand_Having with Flag logic
# df['Brand_Having_Flagged'] = df['Brand_Having'].where(df['Flag'] == 1)
# brand_having_ohe = pd.get_dummies(df['Brand_Having_Flagged']).fillna(0).astype(int)

# # Step 3: Create pivot for aspects
# aspect_pivot = df.pivot_table(
#     index=['review_id', df.index],
#     columns='Aspect',
#     values='Avg rating',
#     fill_value=0
# ).reset_index(level=1, drop=True).reset_index()

# # Step 4: Merge everything
# # Attach original index before pivot
# df_temp = df.copy()
# df_temp['row_id'] = df_temp.index

# # Recreate pivot with row_id
# aspect_pivot = df_temp.pivot_table(
#     index=['review_id', 'row_id'],
#     columns='Aspect',
#     values='Avg rating',
#     fill_value=0
# ).reset_index()

# # Merge Brand + Brand_Having using row_id
# brand_data = df_temp[['row_id']].join(brand_ohe).join(brand_having_ohe)

# final_df = pd.merge(aspect_pivot, brand_data, on='row_id', how='left')

# # Drop helper column
# final_df = final_df.drop(columns=['row_id'])

# # Add star_rating
# star_rating_df = df.groupby('review_id')['star_rating'].first().reset_index()
# final_df = pd.merge(final_df, star_rating_df, on='review_id', how='left')

# # Step 5: Reorder columns
# brand_cols = brand_ohe.columns.tolist()
# brand_having_cols = brand_having_ohe.columns.tolist()

# aspect_cols = [col for col in final_df.columns
#                if col not in ['review_id', 'star_rating'] + brand_cols + brand_having_cols]
# #
# final_df = final_df[['review_id', 'star_rating'] + brand_cols + brand_having_cols + aspect_cols]

# # Step 6: Save
# output_file = "final_transformed_output.csv"
# final_df.to_csv(output_file, index=False)

# from google.colab import files
# files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Old do not run this

In [ ]:
# # Step 1: Rename columns (if needed for consistency)
# df = df.rename(columns={
#     'cluster_level_bottom': 'Aspect',
#     'avg_score': 'Avg rating'
# })

# # Step 2: One-hot encode Brand
# brand_ohe = pd.get_dummies(df['Brand'])

# # Step 3: Create pivot for aspects (each row keeps only its aspect value)
# aspect_pivot = df.pivot_table(
#     index=['review_id', df.index],  # keep rows separate
#     columns='Aspect',
#     values='Avg rating',
#     fill_value=0
# ).reset_index(level=1, drop=True).reset_index()

# # Step 4: Merge Brand encoding
# final_df = pd.concat([aspect_pivot, brand_ohe], axis=1)

# # Step 5: Reorder columns (Review Id first, then brands, then aspects)
# brand_cols = brand_ohe.columns.tolist()
# aspect_cols = [col for col in final_df.columns if col not in ['Review Id'] + brand_cols]

# final_df = final_df[['review_id'] + brand_cols + aspect_cols]

# # Step 6: Save
# output_file = "final_transformed_output.csv"
# final_df.to_csv(output_file, index=False)

# from google.colab import files
# files.download(output_file)

KeyError: 'Brand'

Run this


In [ ]:
# # -------------------------------
# # STEP 7: Group by review_id properly
# # -------------------------------

# # Fix duplicate column issue (VERY IMPORTANT)
# final_df = final_df.loc[:, ~final_df.columns.duplicated()]

# # Define columns
# review_col = 'review_id'
# brand_cols = brand_ohe.columns.tolist()

# # Aspect columns = everything except review_id + brand columns
# aspect_cols = [col for col in final_df.columns if col not in [review_col] + brand_cols]

# # -------------------------------
# # Combine brand + aspect columns
# # -------------------------------
# feature_cols = brand_cols + aspect_cols

# # -------------------------------
# # Replace 0 with NaN (ignore missing values)
# # -------------------------------
# temp_df = final_df.copy()
# temp_df[feature_cols] = temp_df[feature_cols].replace(0, np.nan)

# # -------------------------------
# # Group and compute mean (for BOTH brand + aspects)
# # -------------------------------
# grouped_df = temp_df.groupby(review_col)[feature_cols].mean().reset_index()

# # -------------------------------
# # Fill NaN back to 0
# # -------------------------------
# grouped_df[feature_cols] = grouped_df[feature_cols].fillna(0)

# # -------------------------------
# # Save output
# # -------------------------------
# output_file = "final_grouped_output.csv"
# grouped_df.to_csv(output_file, index=False)

# from google.colab import files
# files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# # -------------------------------
# # STEP 7: Group by review_id properly
# # -------------------------------

# # Fix duplicate column issue (VERY IMPORTANT)
# final_df = final_df.loc[:, ~final_df.columns.duplicated()]

# # Define column
# review_col = 'review_id'

# # -------------------------------
# # Take ALL columns except review_id
# # -------------------------------
# feature_cols = [col for col in final_df.columns if col != review_col]

# # -------------------------------
# # Replace 0 with NaN (ignore missing values)
# # -------------------------------
# temp_df = final_df.copy()
# temp_df[feature_cols] = temp_df[feature_cols].replace(0, np.nan)

# # -------------------------------
# # Group and compute mean
# # -------------------------------
# grouped_df = temp_df.groupby(review_col)[feature_cols].mean().reset_index()

# # -------------------------------
# # Fill NaN back to 0
# # -------------------------------
# grouped_df[feature_cols] = grouped_df[feature_cols].fillna(0)

# # -------------------------------
# # Save output
# # -------------------------------
# output_file = "final_grouped_output.csv"
# grouped_df.to_csv(output_file, index=False)

# from google.colab import files
# files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>